# Energy Reconstruction Using CNN - Both Charges and Cos(Zenith)

In [12]:
import numpy as np
import tensorflow as tf
import os
import matplotlib.pyplot as plt

from datetime import datetime

from tensorflow import keras
from keras import layers
from keras import models
from keras import callbacks

#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Conv2D, Flatten
#from tensorflow.keras.callbacks import ModelCheckpoint

from data_tools import load_preprocessed, dataPrep, nameModel

simPrefix = os.getcwd()+'\\simdata'

In [13]:
tf.__version__

'2.8.0'

## Model Design

In [14]:
# Name for model
name = 'classify'
i = 0
while(os.path.exists('untrainedModels/{}.h5'.format(name+str(i)))):
    i = i + 1
name = name+str(i)

# Data preparation: no merging of charge (q), no time layers included (t=False), data normalized from 0-1
prep = {'q':None, 't':False, 'normed':True, 'reco':'plane', 'cosz':False}
print(name)

classify3


In [15]:
# Create model using functional API for multiple inputs
charge_input=keras.Input(shape=(10,10,2,))

conv1_layer = layers.Conv2D(64,kernel_size=3,padding='same',activation='relu')(charge_input)

#conv2_layer = layers.Conv2D(32,kernel_size=3,padding='same',activation='relu')(conv1_layer)

#conv3_layer = layers.Conv2D(16, kernel_size=3, padding='same',activation="relu")(conv2_layer)

#flat_layer = layers.Flatten()(conv3_layer)
flat_layer = layers.Flatten()(conv1_layer)
zenith_input=keras.Input(shape=(1,))
concat_layer = layers.Concatenate()([flat_layer,zenith_input])

#dense1_layer = layers.Dense(256,activation='relu')(concat_layer)
dense2_layer = layers.Dense(128,activation='relu')(concat_layer)

#dense2_layer = layers.Dense(256,activation='relu')(dense1_layer)

#dense3_layer = layers.Dense(256,activation="relu")(dense2_layer)

#Composition has 2 possibilities
#output = layers.Dense(2)(dense3_layer)
output = layers.Dense(2)(dense2_layer)

model = models.Model(inputs=[charge_input,zenith_input],outputs=output,name=name)

model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), optimizer='adam', metrics=['accuracy'])

In [16]:
sample_model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(10,10,2)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(2)
])
sample_model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [17]:
model.summary()

Model: "classify3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 10, 10, 2)]  0           []                               
                                                                                                  
 flatten_1 (Flatten)            (None, 200)          0           ['input_3[0][0]']                
                                                                                                  
 input_4 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 concatenate_1 (Concatenate)    (None, 201)          0           ['flatten_1[0][0]',              
                                                                  'input_4[0][0]']        

In [18]:
model.save('untrainedModels/%s.h5'% name)
np.save('untrainedModels/%s.npy' % name,prep)